In [0]:
%sql
/*
Descrição:
Este trabalho tem como finalidade a construção de um pipeline de dados em nuvem, utilizando principalmente SQL, a partir de uma base de dados simulada para vendas de um e-commerce, base é chamada de Amazon Sales Dataset e foi extraída no Kaggle (https://www.kaggle.com/datasets/rohiteng/amazon-sales-dataset).

O MVP busca desenvolver uma estrutura de dados confiável e escalável para analisar o desempenho das vendas de um e-commerce, permitindo identificar padrões relacionados à receita, produtos, clientes, descontos, métodos de pagamento, vendedores e status dos pedidos. Transformando dados transacionais brutos em dados estruturados e documentados, aplicando processos de ingestão, transformação, validação e modelagem dimensional que possibilitem análises de negócio de forma consistente e reprodutível.

Objetivo:
Desenvolver um pipeline de dados em nuvem que permita a ingestão, transformação e modelagem de dados para análise de vendas de um e-commerce, com o objetivo de identificar padrões e tendências que possam auxiliar na tomada de decisões estratégicas. Deseja-se responder as seguintes perguntas ao longo do projeto:

1. Como a receita e o volume de pedidos evoluem ao longo do tempo?
2. Quais categorias, marcas e produtos apresentam maior volume e valor de vendas?
3. Como diferentes níveis de desconto estão associados ao volume e ao valor das vendas?
4. Como preço, quantidade, desconto, impostos e frete se relacionam com o valor total das transações?
5. Como as vendas se distribuem entre clientes e localizações geográficas?
6. Como os diferentes status dos pedidos se distribuem e qual é sua relação com o valor das transações?
7. Quais métodos de pagamento são utilizados e como eles se relacionam com o volume e valor das transações?
8. Como o desempenho das vendas se distribui entre os vendedores?

Limitações:
1. O dataset possui dados sintéticos, não representando necessariamente o comportamento real de consumidores ou operações de comércio eletrônico.
2. Não há garantia de que as relações entre clientes, produtos, categorias, marcas e localizações representem relações reais.
3. O campo TotalAmount representa o valor informado na fonte e não necessariamente receita reconhecida contabilmente.
4. As análises de desconto identificam associações entre variáveis, não relações causais.
5. Inconsistências identificadas na fonte serão documentadas e tratadas de forma explícita no pipeline.
*/


In [0]:
/* Criando os Schemas a serem utilizados ao longo do projeto.

A arquitetura Medallion segue a seguinte estrutura:
Bronze	Dados brutos obtidos através de arquivo .csv;
Silver	Dados tratados após observação das limitações vindas da planilha;
Gold	Dados prontos para análise
 */


In [0]:
CREATE CATALOG IF NOT EXISTS amazon_sales;

In [0]:
SHOW CATALOGS;

catalog
amazon_sales
samples
system
workspace


In [0]:
CREATE SCHEMA IF NOT EXISTS amazon_sales.bronze;

In [0]:
CREATE SCHEMA IF NOT EXISTS amazon_sales.silver;

In [0]:
CREATE SCHEMA IF NOT EXISTS amazon_sales.gold;

In [0]:
CREATE VOLUME IF NOT EXISTS amazon_sales.bronze.raw_files;

In [0]:
LIST '/Volumes/amazon_sales/bronze/raw_files/';

path,name,size,modification_time
/Volumes/amazon_sales/bronze/raw_files/Amazon.csv,Amazon.csv,17377579,1789695556000


In [0]:
CREATE OR REPLACE TABLE amazon_sales.bronze.sales_raw AS
SELECT *
FROM read_files(
    '/Volumes/amazon_sales/bronze/raw_files/Amazon.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);

num_affected_rows,num_inserted_rows


**Os códigos abaixo foram utilizados para análise exploratória dos dados**

In [0]:
SELECT COUNT(*) AS total_registros
FROM amazon_sales.bronze.sales_raw;

total_registros
100000


In [0]:
/* Primeira visualização da base de dados */
SELECT * FROM bronze.sales_raw LIMIT 10;


OrderID,OrderDate,CustomerID,CustomerName,ProductID,ProductName,Category,Brand,Quantity,UnitPrice,Discount,Tax,ShippingCost,TotalAmount,PaymentMethod,OrderStatus,City,State,Country,SellerID
ORD0000001,2023-01-31,CUST001504,Vihaan Sharma,P00014,Drone Mini,Books,BrightLux,3,106.59,0.0,0.0,0.09,319.86,Debit Card,Delivered,Washington,DC,India,SELL01967
ORD0000002,2023-12-30,CUST000178,Pooja Kumar,P00040,Microphone,Home & Kitchen,UrbanStyle,1,251.37,0.05,19.1,1.74,259.64,Amazon Pay,Delivered,Fort Worth,TX,United States,SELL01298
ORD0000003,2022-05-10,CUST047516,Sneha Singh,P00044,Power Bank 20000mAh,Clothing,UrbanStyle,3,35.03,0.1,7.57,5.91,108.06,Debit Card,Delivered,Austin,TX,United States,SELL00908
ORD0000004,2023-07-18,CUST030059,Vihaan Reddy,P00041,Webcam Full HD,Home & Kitchen,Zenith,5,33.58,0.15,11.42,5.53,159.66,Cash on Delivery,Delivered,Charlotte,NC,India,SELL01164
ORD0000005,2023-02-04,CUST048677,Aditya Kapoor,P00029,T-Shirt,Clothing,KiddoFun,2,515.64,0.25,38.67,9.23,821.36,Credit Card,Cancelled,San Antonio,TX,Canada,SELL01411
ORD0000006,2022-12-31,CUST042705,Karan Sharma,P00023,Cookware Set,Books,ReadMore,4,449.73,0.0,215.87,2.74,2017.53,UPI,Delivered,Los Angeles,CA,United States,SELL01494
ORD0000007,2024-09-20,CUST037667,Aarav Verma,P00030,Dress Shirt,Clothing,UrbanStyle,2,219.81,0.2,28.14,14.97,394.81,UPI,Delivered,Chicago,IL,Australia,SELL01676
ORD0000008,2022-11-10,CUST031165,Rohit Kumar,P00028,Jeans,Toys & Games,KiddoFun,2,306.51,0.05,29.12,6.24,617.73,Debit Card,Pending,Denver,CO,India,SELL00510
ORD0000009,2024-06-26,CUST026965,Aman Kapoor,P00031,Kids Toy Car,Sports & Outdoors,Apex,4,146.09,0.0,46.75,7.03,638.14,Debit Card,Delivered,Washington,DC,United States,SELL01895
ORD0000010,2020-05-01,CUST029472,Aarav Reddy,P00001,Wireless Earbuds,Clothing,Apex,2,278.21,0.1,60.09,4.88,565.75,Credit Card,Delivered,Houston,TX,United States,SELL01584


In [0]:
/* Verificando quantidade total de linhas: 100.000 */
SELECT COUNT(*) AS total_records
FROM amazon_sales.bronze.sales_raw;

total_records
100000


In [0]:
/*
Verificando o formato de cada coluna:
OrderID: texto
OrderDate: data
CustomerID: texto
CustomerName:texto
ProductID: texto
ProductName: texto
Category: texto
Brand: texto
Quantity: int
UnitPrice: double
Discount: double
Tax: double
ShippingCost: double
TotalAmount: double
PaymentMethod: texto
OrderStatus: texto
City: texto
State: texto
Country: texto
SellerID: texto

*/
DESCRIBE amazon_sales.bronze.sales_raw;

col_name,data_type,comment
OrderID,string,null
OrderDate,date,null
CustomerID,string,null
CustomerName,string,null
ProductID,string,null
ProductName,string,null
Category,string,null
Brand,string,null
Quantity,int,null
UnitPrice,double,null


In [0]:
/* Verificando valores nulos na base original: não existem nulos nesta base de dados */
SELECT
    COUNT(*) AS total_records,
    COUNT_IF(OrderID IS NULL) AS null_order_id,
    COUNT_IF(OrderDate IS NULL) AS null_order_date,
    COUNT_IF(CustomerID IS NULL) AS null_customer_id,
    COUNT_IF(ProductID IS NULL) AS null_product_id,
    COUNT_IF(Quantity IS NULL) AS null_quantity,
    COUNT_IF(UnitPrice IS NULL) AS null_unit_price,
    COUNT_IF(Discount IS NULL) AS null_discount,
    COUNT_IF(Tax IS NULL) AS null_tax,
    COUNT_IF(ShippingCost IS NULL) AS null_shipping,
    COUNT_IF(TotalAmount IS NULL) AS null_total_amount,
    COUNT_IF(OrderStatus IS NULL) AS null_status
FROM amazon_sales.bronze.sales_raw;

total_records,null_order_id,null_order_date,null_customer_id,null_product_id,null_quantity,null_unit_price,null_discount,null_tax,null_shipping,null_total_amount,null_status
100000,0,0,0,0,0,0,0,0,0,0,0


In [0]:
/* Verificando se existe duplicidade de OrderID na base bronze: não existe, a quantidade de valores únicos é igual a quantidade total de linhas. */
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT OrderID) AS unique_orders
FROM amazon_sales.bronze.sales_raw;

total_rows,unique_orders
100000,100000


In [0]:
/* Comprovação da unicidade de OrderID */
SELECT
    OrderID,
    COUNT(*) AS occurrences
FROM amazon_sales.bronze.sales_raw
GROUP BY OrderID
HAVING COUNT(*) > 1;

OrderID,occurrences


In [0]:
/* Nesta etapa, vamos entender melhor como estão distribuidos os valores das variáveis categórias

Começando primeiro por Status: */
SELECT
    OrderStatus,
    COUNT(*) AS records
FROM amazon_sales.bronze.sales_raw
GROUP BY OrderStatus
ORDER BY records DESC;

OrderStatus,records
Delivered,74628
Shipped,15192
Pending,4103
Returned,3049
Cancelled,3028


In [0]:
/* Verificando a distribuição de Categorias: */
SELECT
    Category,
    COUNT(*) AS records
FROM amazon_sales.bronze.sales_raw
GROUP BY Category
ORDER BY records DESC;

Category,records
Electronics,16853
Sports & Outdoors,16804
Books,16752
Home & Kitchen,16610
Toys & Games,16542
Clothing,16439


In [0]:
/* Verificando a distribuição de Marcas */
SELECT
    Brand,
    COUNT(*) AS records
FROM amazon_sales.bronze.sales_raw
GROUP BY Brand
ORDER BY records DESC;

Brand,records
ReadMore,10204
FitLife,10147
CoreTech,10127
KiddoFun,10077
Zenith,9990
NexPro,9962
Apex,9937
BrightLux,9907
UrbanStyle,9854
HomeEase,9795


In [0]:
/* Verificando a distribuição de Meios de pagamento: */
SELECT
    PaymentMethod,
    COUNT(*) AS records
FROM amazon_sales.bronze.sales_raw
GROUP BY PaymentMethod
ORDER BY records DESC;

PaymentMethod,records
Credit Card,35038
Debit Card,20024
UPI,15066
Amazon Pay,15017
Net Banking,9927
Cash on Delivery,4928


In [0]:
/* Verificando as variáveis quantitativas e suas estatísticas: */
SELECT
    MIN(Quantity) AS min_quantity,
    MAX(Quantity) AS max_quantity,
    AVG(Quantity) AS avg_quantity,

    MIN(UnitPrice) AS min_unit_price,
    MAX(UnitPrice) AS max_unit_price,
    AVG(UnitPrice) AS avg_unit_price,

    MIN(Discount) AS min_discount,
    MAX(Discount) AS max_discount,
    AVG(Discount) AS avg_discount,

    MIN(Tax) AS min_tax,
    MAX(Tax) AS max_tax,
    AVG(Tax) AS avg_tax,

    MIN(ShippingCost) AS min_shipping,
    MAX(ShippingCost) AS max_shipping,
    AVG(ShippingCost) AS avg_shipping,

    MIN(TotalAmount) AS min_total,
    MAX(TotalAmount) AS max_total,
    AVG(TotalAmount) AS avg_total

FROM amazon_sales.bronze.sales_raw;

min_quantity,max_quantity,avg_quantity,min_unit_price,max_unit_price,avg_unit_price,min_discount,max_discount,avg_discount,min_tax,max_tax,avg_tax,min_shipping,max_shipping,avg_shipping,min_total,max_total,avg_total
1,5,3.0014,5.0,599.99,302.90574839999994,0.0,0.3,0.07422600000000082,0.0,538.46,68.46890159999997,0.0,15.0,7.4066600999999945,4.27,3534.98,918.2564792000002


In [0]:
/* Verificando intervalo temporal: */
SELECT
    MIN(OrderDate) AS first_order,
    MAX(OrderDate) AS last_order
FROM amazon_sales.bronze.sales_raw;

first_order,last_order
2020-01-01,2024-12-29


In [0]:
/* Verificando se a composição do preço condiz com a seguinte fórmula: TotalAmount = Quantity * UnitPrice * (1 - Discount) + Tax + ShippingCost */
SELECT
    COUNT(*) AS total_records,

    COUNT_IF(
        ABS(
            TotalAmount -
            (
                Quantity * UnitPrice * (1 - Discount)
                + Tax
                + ShippingCost
            )
        ) <= 0.01
    ) AS valid_amount,

    COUNT_IF(
        ABS(
            TotalAmount -
            (
                Quantity * UnitPrice * (1 - Discount)
                + Tax
                + ShippingCost
            )
        ) > 0.01
    ) AS invalid_amount

FROM amazon_sales.bronze.sales_raw;

total_records,valid_amount,invalid_amount
100000,100000,0


In [0]:
/* Nota-se que o ProductID possui uma composição heterogênea, onde um mesmo ID pode estar associado a mais de um produto, ou mesmo uma mesma categoria e marca */
SELECT
    ProductID,
    COUNT(DISTINCT ProductName) AS product_names,
    COUNT(DISTINCT Category) AS categories,
    COUNT(DISTINCT Brand) AS brands
FROM amazon_sales.bronze.sales_raw
GROUP BY ProductID
HAVING
    COUNT(DISTINCT ProductName) > 1
    OR COUNT(DISTINCT Category) > 1
    OR COUNT(DISTINCT Brand) > 1;

ProductID,product_names,categories,brands
P00014,1,6,10
P00040,1,6,10
P00044,1,6,10
P00041,1,6,10
P00029,1,6,10
P00023,1,6,10
P00030,1,6,10
P00028,1,6,10
P00031,1,6,10
P00001,1,6,10


In [0]:
/* Nota-se que o CustomerID possui uma composição heterogênea, onde um mesmo ID pode estar associado a mais de um nome, cidade, estado e país */
SELECT
    CustomerID,
    COUNT(DISTINCT CustomerName) AS names,
    COUNT(DISTINCT City) AS cities,
    COUNT(DISTINCT State) AS states,
    COUNT(DISTINCT Country) AS countries
FROM amazon_sales.bronze.sales_raw
GROUP BY CustomerID
HAVING
    COUNT(DISTINCT CustomerName) > 1
    OR COUNT(DISTINCT City) > 1
    OR COUNT(DISTINCT State) > 1
    OR COUNT(DISTINCT Country) > 1;

CustomerID,names,cities,states,countries
CUST001504,3,3,3,2
CUST000178,2,2,2,1
CUST047516,2,2,2,2
CUST030059,6,4,4,2
CUST042705,4,4,2,2
CUST037667,3,3,3,2
CUST031165,3,3,3,3
CUST029472,3,3,2,2
CUST005346,2,2,2,2
CUST046088,2,2,2,2


**Diagnósitco da camada Bronze**
- O dataset apresenta boa completude, unicidade das transações e consistência matemática dos valores financeiros, porém possui inconsistências relacionais entre algumas chaves de negócio e seus atributos descritivos.

In [0]:
/*
- Dimensão	    Resultados	             Tratamento
- Registros	    100.000	                 Manter
- Nulos	        0	                     Nenhum tratamento
- OrderID	    100.000 únicos	         Pode ser chave natural da transação
- OrderStatus	5 valores	             Manter domínio
- Category	    6 valores	             Manter domínio
- Brand	        10 valores	             Manter domínio
- PaymentMethod	6 valores	             Manter domínio
- TotalAmount	Consistente com fórmula	 Manter
- ProductID	    Relações heterogêneas	 Não usar como dimensão 1:1
- CustomerID	Relações heterogêneas	 Não usar como dimensão 1:1

*/

In [0]:
/* Tentando entender melhor a heterogeneidade do ProductID, foi possível notar que o ProductID é composto por uma combinação de ProductName, Category e Brand e que o mais comum é termos um nome de produto com categorias e marcas distintas, algumas inclusive sem relação com o produto como o exemplo abaixo, onde o Wireless Earbuds está na categoria Home & Kitchen */
SELECT
    ProductID,
    ProductName,
    Category,
    Brand,
    COUNT(*) AS registros
FROM amazon_sales.bronze.sales_raw
GROUP BY
    ProductID,
    ProductName,
    Category,
    Brand
ORDER BY ProductID, registros DESC;

ProductID,ProductName,Category,Brand,registros
P00001,Wireless Earbuds,Electronics,ReadMore,50
P00001,Wireless Earbuds,Clothing,KiddoFun,48
P00001,Wireless Earbuds,Electronics,CoreTech,42
P00001,Wireless Earbuds,Home & Kitchen,CoreTech,42
P00001,Wireless Earbuds,Toys & Games,FitLife,42
P00001,Wireless Earbuds,Clothing,Apex,41
P00001,Wireless Earbuds,Toys & Games,HomeEase,41
P00001,Wireless Earbuds,Sports & Outdoors,UrbanStyle,41
P00001,Wireless Earbuds,Sports & Outdoors,KiddoFun,40
P00001,Wireless Earbuds,Books,KiddoFun,40


In [0]:
/* Seguindo o mesmo conceito do ProductID, vemos que o CustomerID é composto por uma combinação de CustomerName, City, State e Country e que o ID se repete em várias combinações com nomes, cidades e estados diferentes. Também foi possível notar que as cidades e estados são do EUA, mas existem diversos países diferentes */
SELECT
    CustomerID,
    CustomerName,
    City,
    State,
    Country,
    COUNT(*) AS registros
FROM amazon_sales.bronze.sales_raw
GROUP BY
    CustomerID,
    CustomerName,
    City,
    State,
    Country
ORDER BY CustomerID, registros DESC;

CustomerID,CustomerName,City,State,Country,registros
CUST000001,Sunita Kumar,Chicago,IL,United States,1
CUST000002,Arjun Singh,Los Angeles,CA,United States,1
CUST000002,Vikas Gupta,Phoenix,AZ,United States,1
CUST000002,Ritika Patel,New York,NY,India,1
CUST000003,Aditya Gupta,San Jose,CA,United States,1
CUST000004,Karan Kapoor,San Diego,CA,Canada,1
CUST000004,Priya Sharma,Houston,TX,United States,1
CUST000004,Sahil Patel,Houston,TX,United States,1
CUST000005,Pooja Kapoor,Chicago,IL,United States,1
CUST000006,Vikas Reddy,Indianapolis,IN,United States,1


In [0]:
/* Devido ao erro não será possível construir uma tabela dato de produto e de cliente. A alternativa será trabalhar no seguinte conceito:
                         GOLD
                           │
                    ┌──────┴──────┐
                    │             │
                FACT_SALES     DIM_DATE
                    │
        ┌───────────┼────────────┐
        │           │            │
     Seller      Payment       Status
        │           │            │
        └───────────┴────────────┘

*/


**Criação da camada Silver**


In [0]:
/* Para tratamento da tabela Silve foi feita a decisão de não manipular ou excluir os dados heterogeneos de CustomerID e ProductID, foi preservado o contexto transacional e esta inconsistência será registrada como uma caractéristica deste dataset */

In [0]:
/* Construindo a tabela Silver ajustando os tipos de dados para facilitar a construção das tabelas */
CREATE OR REPLACE TABLE amazon_sales.silver.sales AS

SELECT
    TRIM(OrderID) AS OrderID,
    OrderDate,

    TRIM(CustomerID) AS CustomerID,
    TRIM(CustomerName) AS CustomerName,

    TRIM(ProductID) AS ProductID,
    TRIM(ProductName) AS ProductName,
    TRIM(Category) AS Category,
    TRIM(Brand) AS Brand,

    Quantity,

    CAST(UnitPrice AS DECIMAL(18,2)) AS UnitPrice,
    CAST(Discount AS DECIMAL(10,4)) AS Discount,
    CAST(Tax AS DECIMAL(18,2)) AS Tax,
    CAST(ShippingCost AS DECIMAL(18,2)) AS ShippingCost,
    CAST(TotalAmount AS DECIMAL(18,2)) AS TotalAmount,

    TRIM(PaymentMethod) AS PaymentMethod,
    TRIM(OrderStatus) AS OrderStatus,

    TRIM(City) AS City,
    TRIM(State) AS State,
    TRIM(Country) AS Country,

    TRIM(SellerID) AS SellerID

FROM amazon_sales.bronze.sales_raw;

num_affected_rows,num_inserted_rows


In [0]:
SELECT
    COUNT(*) AS total_registros
FROM amazon_sales.silver.sales;

total_registros
100000


In [0]:
SELECT *
FROM amazon_sales.silver.sales
LIMIT 10;

OrderID,OrderDate,CustomerID,CustomerName,ProductID,ProductName,Category,Brand,Quantity,UnitPrice,Discount,Tax,ShippingCost,TotalAmount,PaymentMethod,OrderStatus,City,State,Country,SellerID
ORD0000001,2023-01-31,CUST001504,Vihaan Sharma,P00014,Drone Mini,Books,BrightLux,3,106.59,0.0000,0.00,0.09,319.86,Debit Card,Delivered,Washington,DC,India,SELL01967
ORD0000002,2023-12-30,CUST000178,Pooja Kumar,P00040,Microphone,Home & Kitchen,UrbanStyle,1,251.37,0.0500,19.10,1.74,259.64,Amazon Pay,Delivered,Fort Worth,TX,United States,SELL01298
ORD0000003,2022-05-10,CUST047516,Sneha Singh,P00044,Power Bank 20000mAh,Clothing,UrbanStyle,3,35.03,0.1000,7.57,5.91,108.06,Debit Card,Delivered,Austin,TX,United States,SELL00908
ORD0000004,2023-07-18,CUST030059,Vihaan Reddy,P00041,Webcam Full HD,Home & Kitchen,Zenith,5,33.58,0.1500,11.42,5.53,159.66,Cash on Delivery,Delivered,Charlotte,NC,India,SELL01164
ORD0000005,2023-02-04,CUST048677,Aditya Kapoor,P00029,T-Shirt,Clothing,KiddoFun,2,515.64,0.2500,38.67,9.23,821.36,Credit Card,Cancelled,San Antonio,TX,Canada,SELL01411
ORD0000006,2022-12-31,CUST042705,Karan Sharma,P00023,Cookware Set,Books,ReadMore,4,449.73,0.0000,215.87,2.74,2017.53,UPI,Delivered,Los Angeles,CA,United States,SELL01494
ORD0000007,2024-09-20,CUST037667,Aarav Verma,P00030,Dress Shirt,Clothing,UrbanStyle,2,219.81,0.2000,28.14,14.97,394.81,UPI,Delivered,Chicago,IL,Australia,SELL01676
ORD0000008,2022-11-10,CUST031165,Rohit Kumar,P00028,Jeans,Toys & Games,KiddoFun,2,306.51,0.0500,29.12,6.24,617.73,Debit Card,Pending,Denver,CO,India,SELL00510
ORD0000009,2024-06-26,CUST026965,Aman Kapoor,P00031,Kids Toy Car,Sports & Outdoors,Apex,4,146.09,0.0000,46.75,7.03,638.14,Debit Card,Delivered,Washington,DC,United States,SELL01895
ORD0000010,2020-05-01,CUST029472,Aarav Reddy,P00001,Wireless Earbuds,Clothing,Apex,2,278.21,0.1000,60.09,4.88,565.75,Credit Card,Delivered,Houston,TX,United States,SELL01584


In [0]:
DESCRIBE amazon_sales.silver.sales;

col_name,data_type,comment
OrderID,string,null
OrderDate,date,null
CustomerID,string,null
CustomerName,string,null
ProductID,string,null
ProductName,string,null
Category,string,null
Brand,string,null
Quantity,int,null
UnitPrice,"decimal(18,2)",null


In [0]:
/* Verificando consistência da camada Silver */
SELECT
    COUNT(*) AS total_registros,

    COUNT_IF(OrderID IS NULL) AS orderid_nulos,
    COUNT_IF(OrderDate IS NULL) AS orderdate_nulos,
    COUNT_IF(CustomerID IS NULL) AS customerid_nulos,
    COUNT_IF(ProductID IS NULL) AS productid_nulos,

    COUNT_IF(Quantity <= 0) AS quantidade_invalida,
    COUNT_IF(UnitPrice < 0) AS preco_invalido,
    COUNT_IF(Discount < 0 OR Discount > 1) AS desconto_invalido,
    COUNT_IF(Tax < 0) AS imposto_invalido,
    COUNT_IF(ShippingCost < 0) AS frete_invalido,
    COUNT_IF(TotalAmount < 0) AS total_invalido

FROM amazon_sales.silver.sales;

total_registros,orderid_nulos,orderdate_nulos,customerid_nulos,productid_nulos,quantidade_invalida,preco_invalido,desconto_invalido,imposto_invalido,frete_invalido,total_invalido
100000,0,0,0,0,0,0,0,0,0,0


**Criação da camada Gold**

In [0]:
/* Foi notado no dataset que o o OrderID é único, logo ele será o pivô utilizado para criação da tabela Fact_Sales

Seguindo o modelo estrela, teremos o seguinte:
                         ┌──────────────┐
                         │   DIM_DATE   │
                         └──────┬───────┘
                                │
                                │
┌──────────────┐         ┌─────▼──────┐         ┌────────────────┐
│ DIM_PRODUCT  │────────►│ FACT_SALES │◄────────│ DIM_CUSTOMER  │
└──────────────┘         └─────┬──────┘         └────────────────┘
                                │
                 ┌──────────────┼──────────────┐
                 │              │              │
          ┌──────▼─────┐ ┌──────▼──────┐ ┌─────▼────────┐
          │ DIM_SELLER │ │ DIM_STATUS │ │ DIM_PAYMENT  │
          └────────────┘ └─────────────┘ └──────────────┘

Com uma tabela fato - gold.fact_sales e as seguintes dimensões:
gold.dim_date
gold.dim_product
gold.dim_customer
gold.dim_seller
gold.dim_status
gold.dim_payment

Também serão criadas chaves únicas para cada dimensão:
ProductKey
CustomerKey
SellerKey
StatusKey
PaymentKey
DateKey
 */

In [0]:
/* Criação da dimensão DIM_DATE */
CREATE OR REPLACE TABLE amazon_sales.gold.dim_date AS

SELECT DISTINCT
    CAST(date_format(OrderDate, 'yyyyMMdd') AS INT) AS DateKey,
    OrderDate AS Date,
    DAY(OrderDate) AS Day,
    MONTH(OrderDate) AS Month,
    date_format(OrderDate, 'MMMM') AS MonthName,
    QUARTER(OrderDate) AS Quarter,
    YEAR(OrderDate) AS Year,
    dayofweek(OrderDate) AS DayOfWeek,
    date_format(OrderDate, 'EEEE') AS DayName
FROM amazon_sales.silver.sales;

num_affected_rows,num_inserted_rows


In [0]:
SELECT *
FROM amazon_sales.gold.dim_date
ORDER BY Date
LIMIT 10;

DateKey,Date,Day,Month,MonthName,Quarter,Year,DayOfWeek,DayName
20200101,2020-01-01,1,1,January,1,2020,4,Wednesday
20200102,2020-01-02,2,1,January,1,2020,5,Thursday
20200103,2020-01-03,3,1,January,1,2020,6,Friday
20200104,2020-01-04,4,1,January,1,2020,7,Saturday
20200105,2020-01-05,5,1,January,1,2020,1,Sunday
20200106,2020-01-06,6,1,January,1,2020,2,Monday
20200107,2020-01-07,7,1,January,1,2020,3,Tuesday
20200108,2020-01-08,8,1,January,1,2020,4,Wednesday
20200109,2020-01-09,9,1,January,1,2020,5,Thursday
20200110,2020-01-10,10,1,January,1,2020,6,Friday


In [0]:
/* Criação da dimensão DIM_STATUS */
CREATE OR REPLACE TABLE amazon_sales.gold.dim_status AS

SELECT
    ROW_NUMBER() OVER (ORDER BY OrderStatus) AS StatusKey,
    OrderStatus
FROM (
    SELECT DISTINCT OrderStatus
    FROM amazon_sales.silver.sales
);

num_affected_rows,num_inserted_rows


In [0]:
SELECT *
FROM amazon_sales.gold.dim_status
ORDER BY StatusKey;

StatusKey,OrderStatus
1,Cancelled
2,Delivered
3,Pending
4,Returned
5,Shipped


In [0]:
/* Criação da dimensão DIM_PAYMENT */
CREATE OR REPLACE TABLE amazon_sales.gold.dim_payment AS

SELECT
    ROW_NUMBER() OVER (ORDER BY PaymentMethod) AS PaymentKey,
    PaymentMethod
FROM (
    SELECT DISTINCT PaymentMethod
    FROM amazon_sales.silver.sales
);

num_affected_rows,num_inserted_rows


In [0]:
/* Criação da dimensão DIM_PRODUCT */
CREATE OR REPLACE TABLE amazon_sales.gold.dim_seller AS

SELECT
    ROW_NUMBER() OVER (ORDER BY SellerID) AS SellerKey,
    SellerID
FROM (
    SELECT DISTINCT SellerID
    FROM amazon_sales.silver.sales
);

num_affected_rows,num_inserted_rows


In [0]:
/* Para preservar o contexto da transação, a DIM_PRODUCT será construída com a combinação observada de atributos:
ProductID
ProductName
Category
Brand

*/
CREATE OR REPLACE TABLE amazon_sales.gold.dim_product AS

SELECT
    ROW_NUMBER() OVER (
        ORDER BY ProductID, ProductName, Category, Brand
    ) AS ProductKey,

    ProductID,
    ProductName,
    Category,
    Brand

FROM (
    SELECT DISTINCT
        ProductID,
        ProductName,
        Category,
        Brand
    FROM amazon_sales.silver.sales
);

num_affected_rows,num_inserted_rows


In [0]:
/* Será feito algo similar para a DIM_CUSTOMER, visto que o CustomerID não é totalmente deterministico em relação a: CustomerName, City, State, Country */
CREATE OR REPLACE TABLE amazon_sales.gold.dim_customer AS

SELECT
    ROW_NUMBER() OVER (
        ORDER BY CustomerID, CustomerName, City, State, Country
    ) AS CustomerKey,

    CustomerID,
    CustomerName,
    City,
    State,
    Country

FROM (
    SELECT DISTINCT
        CustomerID,
        CustomerName,
        City,
        State,
        Country
    FROM amazon_sales.silver.sales
);

num_affected_rows,num_inserted_rows


In [0]:
/* Criação da tabela fato relacionando cada transação às dimensões */
CREATE OR REPLACE TABLE amazon_sales.gold.fact_sales AS

SELECT

    ROW_NUMBER() OVER (
        ORDER BY s.OrderID
    ) AS SalesKey,

    s.OrderID,

    d.DateKey,
    p.ProductKey,
    c.CustomerKey,
    se.SellerKey,
    st.StatusKey,
    pm.PaymentKey,

    s.Quantity,
    s.UnitPrice,
    s.Discount,
    s.Tax,
    s.ShippingCost,
    s.TotalAmount

FROM amazon_sales.silver.sales s

LEFT JOIN amazon_sales.gold.dim_date d
    ON s.OrderDate = d.Date

LEFT JOIN amazon_sales.gold.dim_product p
    ON s.ProductID = p.ProductID
    AND s.ProductName = p.ProductName
    AND s.Category = p.Category
    AND s.Brand = p.Brand

LEFT JOIN amazon_sales.gold.dim_customer c
    ON s.CustomerID = c.CustomerID
    AND s.CustomerName = c.CustomerName
    AND s.City = c.City
    AND s.State = c.State
    AND s.Country = c.Country

LEFT JOIN amazon_sales.gold.dim_seller se
    ON s.SellerID = se.SellerID

LEFT JOIN amazon_sales.gold.dim_status st
    ON s.OrderStatus = st.OrderStatus

LEFT JOIN amazon_sales.gold.dim_payment pm
    ON s.PaymentMethod = pm.PaymentMethod;

num_affected_rows,num_inserted_rows


In [0]:
SELECT
    COUNT(*) AS total_registros,

    COUNT_IF(DateKey IS NULL) AS date_sem_chave,
    COUNT_IF(ProductKey IS NULL) AS product_sem_chave,
    COUNT_IF(CustomerKey IS NULL) AS customer_sem_chave,
    COUNT_IF(SellerKey IS NULL) AS seller_sem_chave,
    COUNT_IF(StatusKey IS NULL) AS status_sem_chave,
    COUNT_IF(PaymentKey IS NULL) AS payment_sem_chave

FROM amazon_sales.gold.fact_sales;

total_registros,date_sem_chave,product_sem_chave,customer_sem_chave,seller_sem_chave,status_sem_chave,payment_sem_chave
100000,0,0,0,0,0,0


In [0]:
SELECT
    COUNT(*) AS registros,
    COUNT(DISTINCT OrderID) AS pedidos,
    COUNT(*) - COUNT(DISTINCT OrderID) AS duplicidades
FROM amazon_sales.gold.fact_sales;

registros,pedidos,duplicidades
100000,100000,0


In [0]:
/* Verificar se os valores financeiros foram preservados entre as tabelas prata e ouro */
SELECT
    COUNT(*) AS registros_diferentes
FROM amazon_sales.silver.sales s
INNER JOIN amazon_sales.gold.fact_sales f
    ON s.OrderID = f.OrderID
WHERE
    s.Quantity <> f.Quantity
    OR s.UnitPrice <> f.UnitPrice
    OR s.Discount <> f.Discount
    OR s.Tax <> f.Tax
    OR s.ShippingCost <> f.ShippingCost
    OR s.TotalAmount <> f.TotalAmount;

registros_diferentes
0


**Verificaçao de integridade da tabela Gold**

In [0]:
/* Já foi feito o profiling inicial na Bronze, agora vamos fazer o profiling final na Gold */

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT OrderID) AS pedidos_unicos,

    COUNT_IF(OrderID IS NULL) AS orderid_nulos,
    COUNT_IF(DateKey IS NULL) AS datekey_nulos,
    COUNT_IF(ProductKey IS NULL) AS productkey_nulos,
    COUNT_IF(CustomerKey IS NULL) AS customerkey_nulos,
    COUNT_IF(SellerKey IS NULL) AS sellerkey_nulos,
    COUNT_IF(StatusKey IS NULL) AS statuskey_nulos,
    COUNT_IF(PaymentKey IS NULL) AS paymentkey_nulos

FROM amazon_sales.gold.fact_sales;

total_registros,pedidos_unicos,orderid_nulos,datekey_nulos,productkey_nulos,customerkey_nulos,sellerkey_nulos,statuskey_nulos,paymentkey_nulos
100000,100000,0,0,0,0,0,0,0


In [0]:
/* Verificaremos também se há duplicidades nos registros */
SELECT
    OrderID,
    COUNT(*) AS ocorrencias
FROM amazon_sales.gold.fact_sales
GROUP BY OrderID
HAVING COUNT(*) > 1
ORDER BY ocorrencias DESC;

OrderID,ocorrencias


**Avaliação de Qualidade dos dados**

In [0]:
/* Vamos verificar se fórmula do total de vendas se preservou após manipulação ao longo do projeto */
SELECT
    COUNT(*) AS total_registros,

    COUNT_IF(
        ABS(
            TotalAmount -
            (
                Quantity * UnitPrice * (1 - Discount)
                + Tax
                + ShippingCost
            )
        ) <= 0.01
    ) AS calculos_validos,

    COUNT_IF(
        ABS(
            TotalAmount -
            (
                Quantity * UnitPrice * (1 - Discount)
                + Tax
                + ShippingCost
            )
        ) > 0.01
    ) AS calculos_invalidos

FROM amazon_sales.gold.fact_sales;

total_registros,calculos_validos,calculos_invalidos
100000,100000,0


In [0]:
/* Segue catalogação dos dados utilizados durante todo o projeto */
CREATE OR REPLACE TABLE amazon_sales.gold.data_catalog AS

SELECT * FROM VALUES

('fact_sales_analysis','SalesKey','BIGINT','Chave substituta da transação','Identificador técnico','Generated'),
('fact_sales_analysis','OrderID','STRING','Identificador original do pedido','Chave natural da transação','Silver.sales'),
('fact_sales_analysis','DateKey','INT','Chave da dimensão calendário','Relacionamento com dim_date','dim_date'),
('fact_sales_analysis','ProductKey','BIGINT','Chave da dimensão de produto','Relacionamento com dim_product','dim_product'),
('fact_sales_analysis','CustomerKey','BIGINT','Chave da dimensão de cliente','Relacionamento com dim_customer','dim_customer'),
('fact_sales_analysis','SellerKey','BIGINT','Chave da dimensão de vendedor','Relacionamento com dim_seller','dim_seller'),
('fact_sales_analysis','StatusKey','BIGINT','Chave da dimensão de status','Relacionamento com dim_status','dim_status'),
('fact_sales_analysis','PaymentKey','BIGINT','Chave da dimensão de pagamento','Relacionamento com dim_payment','dim_payment'),
('fact_sales_analysis','Quantity','INT','Quantidade de unidades na transação','Valor inteiro positivo','Silver.sales'),
('fact_sales_analysis','UnitPrice','DECIMAL(18,2)','Preço unitário','Valor monetário','Silver.sales'),
('fact_sales_analysis','Discount','DECIMAL(10,4)','Percentual de desconto representado como decimal','Esperado entre 0 e 1','Silver.sales'),
('fact_sales_analysis','Tax','DECIMAL(18,2)','Valor de imposto da transação','Valor monetário','Silver.sales'),
('fact_sales_analysis','ShippingCost','DECIMAL(18,2)','Custo de frete','Valor monetário','Silver.sales'),
('fact_sales_analysis','TotalAmount','DECIMAL(18,2)','Valor total informado na transação','Valor monetário','Silver.sales'),
('fact_sales_analysis','Outlier_TotalAmount','INT','Flag de potencial outlier no valor total','0 = não; 1 = potencial outlier','Derived')

AS t(
    table_name,
    column_name,
    data_type,
    description,
    domain_or_rule,
    lineage
);

num_affected_rows,num_inserted_rows


In [0]:
/* Exibição do catálogo */
SELECT *
FROM amazon_sales.gold.data_catalog
ORDER BY table_name, column_name;

table_name,column_name,data_type,description,domain_or_rule,lineage
fact_sales_analysis,CustomerKey,BIGINT,Chave da dimensão de cliente,Relacionamento com dim_customer,dim_customer
fact_sales_analysis,DateKey,INT,Chave da dimensão calendário,Relacionamento com dim_date,dim_date
fact_sales_analysis,Discount,"DECIMAL(10,4)",Percentual de desconto representado como decimal,Esperado entre 0 e 1,Silver.sales
fact_sales_analysis,OrderID,STRING,Identificador original do pedido,Chave natural da transação,Silver.sales
fact_sales_analysis,Outlier_TotalAmount,INT,Flag de potencial outlier no valor total,0 = não; 1 = potencial outlier,Derived
fact_sales_analysis,PaymentKey,BIGINT,Chave da dimensão de pagamento,Relacionamento com dim_payment,dim_payment
fact_sales_analysis,ProductKey,BIGINT,Chave da dimensão de produto,Relacionamento com dim_product,dim_product
fact_sales_analysis,Quantity,INT,Quantidade de unidades na transação,Valor inteiro positivo,Silver.sales
fact_sales_analysis,SalesKey,BIGINT,Chave substituta da transação,Identificador técnico,Generated
fact_sales_analysis,SellerKey,BIGINT,Chave da dimensão de vendedor,Relacionamento com dim_seller,dim_seller


In [0]:
/* 
Lineage do projeto:
Kaggle / GitHub CSV
        │
        ▼
Amazon.csv
        │
        ▼
amazon_sales.bronze.sales_raw
        │
        │ TRIM
        │ DECIMAL
        │ padronização
        ▼
amazon_sales.silver.sales
        │
        ├───────────────┐
        │               │
        ▼               ▼
   DIMENSÕES        FACT_SALES
        │               │
        └───────┬───────┘
                ▼
      FACT_SALES_ANALYSIS
                │
                ▼
          Análises SQL

As principais alterações ajustadas foram:
Origem	                Destino	                        Transformação
CSV	                    bronze.sales_raw	            Ingestão
bronze.sales_raw	    silver.sales	                TRIM, tipagem e padronização
silver.sales	        dim_date	                    Derivação de atributos de calendário
silver.sales	        dim_product	                    Distinct das combinações observadas
silver.sales	        dim_customer	                Distinct das combinações observadas
silver.sales	        dim_seller	                    Distinct de vendedores
silver.sales	        dim_status	                    Distinct de status
silver.sales	        dim_payment	                    Distinct de pagamentos
silver.sales	        fact_sales	                    Relacionamento com dimensões
fact_sales	            fact_sales_analysis	            Flags de outliers
 */

Para comprovar a implementação do banco de dados, iremos responder as perguntas selecionadas no início do MVP:

In [0]:
/*
1. Como a receita e o volume de pedidos evoluem ao longo do tempo?
2. Quais categorias, marcas e produtos apresentam maior volume e valor de vendas?
3. Como diferentes níveis de desconto estão associados ao volume e ao valor das vendas?
4. Como preço, quantidade, desconto, impostos e frete se relacionam com o valor total das transações?
5. Como as vendas se distribuem entre clientes e localizações geográficas?
6. Como os diferentes status dos pedidos se distribuem e qual é sua relação com o valor das transações?
7. Quais métodos de pagamento são utilizados e como eles se relacionam com o volume e valor das transações?
8. Como o desempenho das vendas se distribui entre os vendedores?
*/

In [0]:
/* Criação de table fato para análises de negócio, com incremento de flags de outliers */
CREATE OR REPLACE TABLE amazon_sales.gold.fact_sales_analysis AS

WITH estatisticas AS (

    SELECT
        percentile_approx(Quantity, 0.25) AS q1_quantity,
        percentile_approx(Quantity, 0.75) AS q3_quantity,

        percentile_approx(UnitPrice, 0.25) AS q1_unitprice,
        percentile_approx(UnitPrice, 0.75) AS q3_unitprice,

        percentile_approx(Discount, 0.25) AS q1_discount,
        percentile_approx(Discount, 0.75) AS q3_discount,

        percentile_approx(Tax, 0.25) AS q1_tax,
        percentile_approx(Tax, 0.75) AS q3_tax,

        percentile_approx(ShippingCost, 0.25) AS q1_shipping,
        percentile_approx(ShippingCost, 0.75) AS q3_shipping,

        percentile_approx(TotalAmount, 0.25) AS q1_total,
        percentile_approx(TotalAmount, 0.75) AS q3_total

    FROM amazon_sales.gold.fact_sales

),

limites AS (

    SELECT

        q1_quantity - 1.5 * (q3_quantity - q1_quantity) AS quantity_lower,
        q3_quantity + 1.5 * (q3_quantity - q1_quantity) AS quantity_upper,

        q1_unitprice - 1.5 * (q3_unitprice - q1_unitprice) AS unitprice_lower,
        q3_unitprice + 1.5 * (q3_unitprice - q1_unitprice) AS unitprice_upper,

        q1_discount - 1.5 * (q3_discount - q1_discount) AS discount_lower,
        q3_discount + 1.5 * (q3_discount - q1_discount) AS discount_upper,

        q1_tax - 1.5 * (q3_tax - q1_tax) AS tax_lower,
        q3_tax + 1.5 * (q3_tax - q1_tax) AS tax_upper,

        q1_shipping - 1.5 * (q3_shipping - q1_shipping) AS shipping_lower,
        q3_shipping + 1.5 * (q3_shipping - q1_shipping) AS shipping_upper,

        q1_total - 1.5 * (q3_total - q1_total) AS total_lower,
        q3_total + 1.5 * (q3_total - q1_total) AS total_upper

    FROM estatisticas

)

SELECT

    f.*,

    CASE
        WHEN f.Quantity < l.quantity_lower
          OR f.Quantity > l.quantity_upper
        THEN 1
        ELSE 0
    END AS Outlier_Quantity,

    CASE
        WHEN f.UnitPrice < l.unitprice_lower
          OR f.UnitPrice > l.unitprice_upper
        THEN 1
        ELSE 0
    END AS Outlier_UnitPrice,

    CASE
        WHEN f.Discount < l.discount_lower
          OR f.Discount > l.discount_upper
        THEN 1
        ELSE 0
    END AS Outlier_Discount,

    CASE
        WHEN f.Tax < l.tax_lower
          OR f.Tax > l.tax_upper
        THEN 1
        ELSE 0
    END AS Outlier_Tax,

    CASE
        WHEN f.ShippingCost < l.shipping_lower
          OR f.ShippingCost > l.shipping_upper
        THEN 1
        ELSE 0
    END AS Outlier_ShippingCost,

    CASE
        WHEN f.TotalAmount < l.total_lower
          OR f.TotalAmount > l.total_upper
        THEN 1
        ELSE 0
    END AS Outlier_TotalAmount

FROM amazon_sales.gold.fact_sales f
CROSS JOIN limites l;

num_affected_rows,num_inserted_rows


In [0]:
/* 1. Como a receita e o volume de pedidos evoluem ao longo do tempo?
Conforme quadro, é possível notar que os tanto o valor transacionado quanto o ticket médio se mantém estabilizados ao longo de tempo, com pequenas variações ao longo dos meses, devido a alguma possível sazonalidade.
*/
SELECT
    d.Year,
    d.Month,
    d.MonthName,

    COUNT(DISTINCT f.OrderID) AS pedidos,

    SUM(f.Quantity) AS unidades,

    ROUND(SUM(f.TotalAmount), 2) AS valor_transacionado,

    ROUND(
        SUM(f.TotalAmount) / COUNT(DISTINCT f.OrderID),
        2
    ) AS ticket_medio

FROM amazon_sales.gold.fact_sales_analysis f

JOIN amazon_sales.gold.dim_date d
    ON f.DateKey = d.DateKey

GROUP BY
    d.Year,
    d.Month,
    d.MonthName

ORDER BY
    d.Year,
    d.Month;

Year,Month,MonthName,pedidos,unidades,valor_transacionado,ticket_medio
2020,1,January,1730,5225,1637069.40,946.28
2020,2,February,1490,4343,1339388.21,898.92
2020,3,March,1685,5050,1587125.66,941.91
2020,4,April,1692,5156,1588347.29,938.74
2020,5,May,1672,5061,1533834.17,917.36
2020,6,June,1633,4883,1500269.29,918.72
2020,7,July,1687,5061,1552981.94,920.56
2020,8,August,1783,5439,1640874.93,920.29
2020,9,September,1679,5023,1554682.21,925.96
2020,10,October,1647,4908,1515403.62,920.10


In [0]:
/* 2. Quais categorias, marcas e produtos apresentam maior volume e valor de vendas?
Categorias: Eletrônicos
Marca: CoreTech
Produto: Desk Organizer

~~ Códigos para responder a questão se encontram abaixo ~~
*/

In [0]:
SELECT
    p.Category,

    COUNT(DISTINCT f.OrderID) AS pedidos,
    SUM(f.Quantity) AS unidades,
    ROUND(SUM(f.TotalAmount), 2) AS valor_transacionado,
    ROUND(AVG(f.TotalAmount), 2) AS ticket_medio

FROM amazon_sales.gold.fact_sales_analysis f

JOIN amazon_sales.gold.dim_product p
    ON f.ProductKey = p.ProductKey

GROUP BY p.Category

ORDER BY valor_transacionado DESC;

Category,pedidos,unidades,valor_transacionado,ticket_medio
Electronics,16853,50696,15584217.18,924.71
Sports & Outdoors,16804,50543,15345571.88,913.21
Books,16752,49981,15261837.01,911.05
Clothing,16439,49523,15253397.50,927.88
Toys & Games,16542,49773,15216684.99,919.88
Home & Kitchen,16610,49624,15163939.36,912.94


In [0]:
SELECT
    p.Brand,

    COUNT(DISTINCT f.OrderID) AS pedidos,
    SUM(f.Quantity) AS unidades,
    ROUND(SUM(f.TotalAmount), 2) AS valor_transacionado

FROM amazon_sales.gold.fact_sales_analysis f

JOIN amazon_sales.gold.dim_product p
    ON f.ProductKey = p.ProductKey

GROUP BY p.Brand

ORDER BY valor_transacionado DESC;

Brand,pedidos,unidades,valor_transacionado
CoreTech,10127,30430,9343720.59
KiddoFun,10077,30168,9324749.55
ReadMore,10204,30421,9278406.63
UrbanStyle,9854,30104,9249027.45
Zenith,9990,30058,9239960.39
Apex,9937,29955,9147604.72
NexPro,9962,29791,9078824.97
FitLife,10147,30121,9061444.68
BrightLux,9907,29616,9056816.11
HomeEase,9795,29476,9045092.83


In [0]:
SELECT
    p.ProductID,
    p.ProductName,
    p.Category,
    p.Brand,

    COUNT(DISTINCT f.OrderID) AS pedidos,
    SUM(f.Quantity) AS unidades,
    ROUND(SUM(f.TotalAmount), 2) AS valor_transacionado

FROM amazon_sales.gold.fact_sales_analysis f

JOIN amazon_sales.gold.dim_product p
    ON f.ProductKey = p.ProductKey

GROUP BY
    p.ProductID,
    p.ProductName,
    p.Category,
    p.Brand

ORDER BY valor_transacionado DESC;

ProductID,ProductName,Category,Brand,pedidos,unidades,valor_transacionado
P00034,Desk Organizer,Clothing,KiddoFun,58,166,58059.53
P00028,Jeans,Sports & Outdoors,Apex,50,156,56329.85
P00047,Memory Card 128GB,Home & Kitchen,HomeEase,53,162,55874.05
P00023,Cookware Set,Books,KiddoFun,43,147,53889.96
P00043,HDMI Cable 2m,Electronics,Zenith,48,154,53200.52
P00019,LED Desk Lamp,Sports & Outdoors,Apex,52,167,52494.29
P00008,4K Monitor,Sports & Outdoors,ReadMore,59,181,52452.39
P00004,USB-C Charger,Sports & Outdoors,Zenith,47,141,52155.04
P00007,Mechanical Keyboard,Home & Kitchen,HomeEase,43,144,51373.57
P00019,LED Desk Lamp,Toys & Games,CoreTech,43,155,51046.67


In [0]:
/* 3. Como diferentes níveis de desconto estão associados ao volume e ao valor das vendas? 
Nota-se que pedidos com 10% de desconto possuem tanto maior valor de venda, quanto maior quantidade de transação e ticket médio.
*/
SELECT

    CASE
        WHEN Discount = 0 THEN '0%'
        WHEN Discount <= 0.10 THEN '0-10%'
        WHEN Discount <= 0.20 THEN '10-20%'
        WHEN Discount <= 0.30 THEN '20-30%'
        ELSE '30%+'
    END AS faixa_desconto,

    COUNT(*) AS pedidos,
    SUM(Quantity) AS unidades,
    ROUND(SUM(TotalAmount), 2) AS valor_transacionado,
    ROUND(AVG(TotalAmount), 2) AS ticket_medio,
    ROUND(AVG(Discount) * 100, 2) AS desconto_medio_percentual

FROM amazon_sales.gold.fact_sales_analysis

GROUP BY
    CASE
        WHEN Discount = 0 THEN '0%'
        WHEN Discount <= 0.10 THEN '0-10%'
        WHEN Discount <= 0.20 THEN '10-20%'
        WHEN Discount <= 0.30 THEN '20-30%'
        ELSE '30%+'
    END

ORDER BY
    faixa_desconto;


faixa_desconto,pedidos,unidades,valor_transacionado,ticket_medio,desconto_medio_percentual
0%,40246,120977,39759304.30,987.91,0.00
0-10%,34799,104281,32145250.64,923.74,7.14
10-20%,17980,53880,14752593.80,820.50,17.21
20-30%,6975,21002,5168499.18,741.00,26.43


In [0]:
/* 4. Como preço, quantidade, desconto, impostos e frete se relacionam com o valor total das transações?
Das variáveis observadas, as que possuem maior correlação são o preço e taxas, visto que elas compõe a maior parte do valor total. Outro ponto interessante de notar é que o desconto possui uma correlação negativa com as vendas, isto porque a base possui maior parte de vendas sem desconto e maior faixa de desconto em até 10% do valor.

*/


In [0]:
SELECT

    ROUND(AVG(Quantity), 2) AS quantidade_media,

    ROUND(AVG(UnitPrice), 2) AS preco_medio,

    ROUND(AVG(Discount) * 100, 2) AS desconto_medio_percentual,

    ROUND(AVG(Tax), 2) AS imposto_medio,

    ROUND(AVG(ShippingCost), 2) AS frete_medio,

    ROUND(AVG(TotalAmount), 2) AS total_medio,

    ROUND(SUM(TotalAmount), 2) AS total_transacionado

FROM amazon_sales.gold.fact_sales_analysis;

quantidade_media,preco_medio,desconto_medio_percentual,imposto_medio,frete_medio,total_medio,total_transacionado
3.0,302.91,7.42,68.47,7.41,918.26,91825647.92


In [0]:
SELECT

    corr(Quantity, TotalAmount) AS corr_quantity_total,

    corr(UnitPrice, TotalAmount) AS corr_price_total,

    corr(Discount, TotalAmount) AS corr_discount_total,

    corr(Tax, TotalAmount) AS corr_tax_total,

    corr(ShippingCost, TotalAmount) AS corr_shipping_total

FROM amazon_sales.gold.fact_sales_analysis;

corr_quantity_total,corr_price_total,corr_discount_total,corr_tax_total,corr_shipping_total
0.5949283533574444,0.7158078950562728,-0.10808814397667924,0.7793990026994994,0.039349348992409945


In [0]:
/* 5. Como as vendas se distribuem entre clientes e localizações geográficas?
Para responder esta questão, é necessário pontuar um dos erros deste dataset, onde os estados e cidades são do EUA, porém existem diversos países ao longo da base. Considerando este obstaculo, conisderaremos apenas a relação com clientes e localização nos estados.
O estado com mais vendas foi o Texas e nota-se que os clientes não possuem tantas vendas, visto que os maiores possuem apenas duas compras ao longo dos anos.
*/ 

In [0]:
/* Estado */
SELECT
    c.Country,
    c.State,

    COUNT(DISTINCT f.OrderID) AS pedidos,
    COUNT(DISTINCT c.CustomerID) AS clientes,
    ROUND(SUM(f.TotalAmount), 2) AS valor_transacionado

FROM amazon_sales.gold.fact_sales_analysis f

JOIN amazon_sales.gold.dim_customer c
    ON f.CustomerKey = c.CustomerKey

GROUP BY
    c.Country,
    c.State

ORDER BY valor_transacionado DESC;

Country,State,pedidos,clientes,valor_transacionado
United States,TX,17334,14662,15897488.53
United States,CA,14014,12277,12783662.94
India,TX,3797,3648,3503209.66
United States,NC,3606,3467,3311877.73
United States,PA,3543,3409,3287459.93
United States,FL,3594,3479,3281693.36
United States,IL,3507,3402,3268618.39
United States,IN,3469,3364,3246583.55
United States,CO,3462,3334,3240143.04
United States,OH,3521,3392,3237947.84


In [0]:
/* Clientes */
SELECT
    c.CustomerID,
    c.CustomerName,

    COUNT(DISTINCT f.OrderID) AS pedidos,
    SUM(f.Quantity) AS unidades,
    ROUND(SUM(f.TotalAmount), 2) AS valor_transacionado,
    ROUND(AVG(f.TotalAmount), 2) AS ticket_medio

FROM amazon_sales.gold.fact_sales_analysis f

JOIN amazon_sales.gold.dim_customer c
    ON f.CustomerKey = c.CustomerKey

GROUP BY
    c.CustomerID,
    c.CustomerName

ORDER BY valor_transacionado DESC;

CustomerID,CustomerName,pedidos,unidades,valor_transacionado,ticket_medio
CUST010696,Pooja Patel,2,10,5436.04,2718.02
CUST035973,Neha Sharma,2,9,5124.43,2562.22
CUST009614,Vikas Mehta,2,9,5003.65,2501.83
CUST018720,Arjun Kapoor,2,9,4773.18,2386.59
CUST022824,Pooja Joshi,2,8,4755.48,2377.74
CUST008882,Sneha Reddy,2,9,4730.47,2365.24
CUST022031,Priya Kapoor,2,7,4266.02,2133.01
CUST031449,Vivaan Kapoor,2,8,4253.27,2126.64
CUST005523,Sahil Patel,2,10,4239.18,2119.59
CUST047013,Arjun Kumar,2,8,4222.62,2111.31


In [0]:
/*6. Como os diferentes status dos pedidos se distribuem e qual é sua relação com o valor das transações?*/

In [0]:
SELECT
    s.OrderStatus,

    COUNT(DISTINCT f.OrderID) AS pedidos,

    ROUND(
        COUNT(DISTINCT f.OrderID) * 100.0 /
        SUM(COUNT(DISTINCT f.OrderID)) OVER (),
        2
    ) AS percentual_pedidos,

    SUM(f.Quantity) AS unidades,

    ROUND(SUM(f.TotalAmount), 2) AS valor_transacionado,

    ROUND(AVG(f.TotalAmount), 2) AS ticket_medio

FROM amazon_sales.gold.fact_sales_analysis f

JOIN amazon_sales.gold.dim_status s
    ON f.StatusKey = s.StatusKey

GROUP BY s.OrderStatus

ORDER BY pedidos DESC;

OrderStatus,pedidos,percentual_pedidos,unidades,valor_transacionado,ticket_medio
Delivered,74628,74.63,223759,68372677.10,916.18
Shipped,15192,15.19,45660,14083628.99,927.04
Pending,4103,4.10,12338,3737556.98,910.93
Returned,3049,3.05,9141,2780662.46,911.99
Cancelled,3028,3.03,9242,2851122.39,941.59


In [0]:

/*7. Quais métodos de pagamento são utilizados e como eles se relacionam com o volume e valor das transações?*/


In [0]:
SELECT
    p.PaymentMethod,

    COUNT(DISTINCT f.OrderID) AS pedidos,

    SUM(f.Quantity) AS unidades,

    ROUND(SUM(f.TotalAmount), 2) AS valor_transacionado,

    ROUND(AVG(f.TotalAmount), 2) AS ticket_medio,

    ROUND(
        SUM(f.TotalAmount) * 100.0 /
        SUM(SUM(f.TotalAmount)) OVER (),
        2
    ) AS participacao_valor_percentual

FROM amazon_sales.gold.fact_sales_analysis f

JOIN amazon_sales.gold.dim_payment p
    ON f.PaymentKey = p.PaymentKey

GROUP BY p.PaymentMethod

ORDER BY valor_transacionado DESC;

PaymentMethod,pedidos,unidades,valor_transacionado,ticket_medio,participacao_valor_percentual
Credit Card,35038,105046,32122158.69,916.78,34.98
Debit Card,20024,60242,18538678.53,925.82,20.19
UPI,15066,45218,13896028.55,922.34,15.13
Amazon Pay,15017,45099,13697498.42,912.13,14.92
Net Banking,9927,29699,9055674.57,912.23,9.86
Cash on Delivery,4928,14836,4515609.16,916.32,4.92


In [0]:

/*8. Como o desempenho das vendas se distribui entre os vendedores?*/

In [0]:
SELECT
    s.SellerID,

    COUNT(DISTINCT f.OrderID) AS pedidos,

    SUM(f.Quantity) AS unidades,

    ROUND(SUM(f.TotalAmount), 2) AS valor_transacionado,

    ROUND(AVG(f.TotalAmount), 2) AS ticket_medio

FROM amazon_sales.gold.fact_sales_analysis f

JOIN amazon_sales.gold.dim_seller s
    ON f.SellerKey = s.SellerKey

GROUP BY s.SellerID

ORDER BY valor_transacionado DESC;

SellerID,pedidos,unidades,valor_transacionado,ticket_medio
SELL00806,67,234,71283.67,1063.94
SELL00440,61,197,71062.59,1164.96
SELL01225,61,204,70785.92,1160.42
SELL00709,69,223,70491.92,1021.62
SELL00221,65,211,70446.50,1083.79
SELL01122,67,217,69469.40,1036.86
SELL00994,61,199,69400.06,1137.71
SELL00536,70,220,69248.00,989.26
SELL00371,65,203,69118.88,1063.37
SELL00812,68,212,68591.86,1008.70


In [0]:
SELECT
    s.SellerID,

    COUNT(DISTINCT f.OrderID) AS pedidos,

    ROUND(SUM(f.TotalAmount), 2) AS valor_transacionado,

    ROUND(
        SUM(
            CASE
                WHEN st.OrderStatus IN ('Cancelled', 'Returned')
                THEN f.TotalAmount
                ELSE 0
            END
        ),
        2
    ) AS valor_cancelado_ou_devolvido

FROM amazon_sales.gold.fact_sales_analysis f

JOIN amazon_sales.gold.dim_seller s
    ON f.SellerKey = s.SellerKey

JOIN amazon_sales.gold.dim_status st
    ON f.StatusKey = st.StatusKey

GROUP BY s.SellerID

ORDER BY valor_transacionado DESC;

SellerID,pedidos,valor_transacionado,valor_cancelado_ou_devolvido
SELL00806,67,71283.67,7833.86
SELL00440,61,71062.59,7206.49
SELL01225,61,70785.92,4223.49
SELL00709,69,70491.92,7445.33
SELL00221,65,70446.50,4299.78
SELL01122,67,69469.40,7867.32
SELL00994,61,69400.06,3322.28
SELL00536,70,69248.00,5351.53
SELL00371,65,69118.88,3915.92
SELL00812,68,68591.86,5423.42
